# **Inverse Modeling of Reservoir Geometry (inverse thickness and z_depth)**
- ### **Condition**: Noisy $S_g$ (time-lapse seismic monitoring)
- ### **Condition**: sparse oberserving wells
- ### **Condition**: Noisy geodata observations

In [1]:
%env CUDA_VISIBLE_DEVICES = 3
import torch
import numpy as np
import matplotlib.pyplot as plt
from functools import partial
import sys
sys.path.append("../../..")
from ConditionalDiffusionGeneration.src.guided_diffusion.unet import create_model
from ConditionalDiffusionGeneration.src.guided_diffusion.condition_methods import get_conditioning_method
from ConditionalDiffusionGeneration.src.guided_diffusion.measurements import get_noise, get_operator
from ConditionalDiffusionGeneration.src.guided_diffusion.gaussian_diffusion import create_sampler
from einops import rearrange

torch.manual_seed(66)
np.random.seed(66)
if torch.cuda.is_available():
    dev = "cuda"
else:
    dev = "cpu"

device = torch.device(dev)
print(device)

def nRMSE(x, x_hat):
    return np.mean(np.sqrt((x - x_hat) ** 2) / np.sqrt(x ** 2))

def RMSE(x, x_hat):
    return np.sqrt((x - x_hat) ** 2).mean()

def nMAE(x, x_hat):
    return np.mean(np.abs(x - x_hat) / np.abs(x))

env: CUDA_VISIBLE_DEVICES=3
cuda


### Load Unconditional Model

In [2]:
u_net_model = create_model(image_size=128,
                           num_channels= 64,
                           num_res_blocks= 2,
                           num_heads=8,
                           num_head_channels=32,
                           attention_resolutions="32,16,8",
                           channel_mult="1,1,2,2,4,4",
                           learn_sigma=True,
                           model_path='../../../UnconditionalDiffusionTraining_and_Generation/output/logs_topography_unstructured/ema.pt'
                        )

u_net_model.to(device)
u_net_model.eval()
print('loaded unet')

loaded unet


# ***Multi source observations***
## CO2 seismic monitoring and geo seismic monitoring

In [3]:
# 0% error: [0., 0., 0., 0.]
# 1% error: [0., 0., 0.3, 8]
# 5% error: [0., 0., 1.5, 40]
# 10% error: [0., 0., 3, 80]

# query_coords_list = [[1613, 1991], [1650.208, 4182.751], [1600, 1500], [1600, 2500], [1000, 2000], [2100, 2000]]  # sparse 6

query_coords_list = [[800, 500], [800, 1500], [800, 2500], [800, 3500], [800, 4500], [800, 5500],
                     [1600, 500], [1600, 1500], [1600, 2500], [1600, 3500], [1600, 4500], [1600, 5500],
                     [2400, 500], [2400, 1500], [2400, 2500], [2400, 3500], [2400, 4500], [2400, 5500],]  # sparse 18

# query_coords_list = [[640, 500], [640, 1500], [640, 2500], [640, 3500], [640, 4500], [640, 5500],
#                      [1280, 500], [1280, 1500], [1280, 2500], [1280, 3500], [1280, 4500], [1280, 5500],
#                      [1920, 500], [1920, 1500], [1920, 2500], [1920, 3500], [1920, 4500], [1920, 5500],
#                      [2560, 500], [2560, 1500], [2560, 2500], [2560, 3500], [2560, 4500], [2560, 5500],]  # sparse 24

operator =  get_operator(name='topography_co2seismic_and_geoseismic',
                        device=device,
                        ckpt_path='../../../ConditionalNeuralField/output_topography_unstructured/all3/checkpoint.pt',
                        norm_record_path='../../../ConditionalNeuralField/output_topography_unstructured/all3/normalizer_params.pt',
                        simdata_path='../../../Dataset/Topography_unstructured_all3.npy',
                        simdata_idx=1998,
                        # query_coords=query_coords_list,
                        time_start_idx=0,
                        time_end_idx=128,
                        time_stride=1,
                        noise_level=[0., 0.05, 1.5, 40]
                        )

# operator =  get_operator(name='topography_geoseismic',
#                         device=device,
#                         ckpt_path='../../../ConditionalNeuralField/output_topography_unstructured/all3/checkpoint.pt',
#                         norm_record_path='../../../ConditionalNeuralField/output_topography_unstructured/all3/normalizer_params.pt',
#                         simdata_path='../../../Dataset/Topography_unstructured_all3.npy',
#                         simdata_idx=1998,
#                         time_start_idx=0,
#                         time_end_idx=128,
#                         time_stride=1,
#                         noise_level=[0., 0., 1.5, 40]
#                         )

# plt.imshow(np.zeros((3200, 5900)))
# for y,x in query_coords_list:
#     plt.scatter(x, y, c='y', s=50, marker='x')

[**Full case**] Using the full time range from 0 to 128 with stride 1.
Simulation data loaded, with shape (128, 7720, 4)
Unstructured coordinates loaded, with shape (7720, 2)


In [ ]:
noiser = get_noise(sigma=0.0, name='gaussian')
cond_method = get_conditioning_method(operator=operator, noiser=noiser, name='ps_adam', scale=1e-2)    # 1e-2
measurement_cond_fn = partial(cond_method.conditioning)

sampler = create_sampler(sampler='ddpm',
                         steps=1000,
                         noise_schedule="cosine",
                         model_mean_type="epsilon",
                         model_var_type="learned_range",
                         dynamic_threshold=False,
                         clip_denoised=True,
                         rescale_timesteps=False,
                         timestep_respacing="")

sample_fn = partial(sampler.p_sample_loop, model=u_net_model, measurement_cond_fn=measurement_cond_fn)

In [ ]:
no_of_samples = 10
time_length = 128
latent_size = 384

x_start = torch.randn(no_of_samples, 1, time_length, latent_size, device=device)
measurement_ref = operator.measurement()
samples = [sample_fn(x_start=x_start[i:i+1], measurement=measurement_ref, record=False, save_root=None) for i in range(x_start.shape[0])]

In [7]:
# gen_latents = torch.cat([samples[i] for i in [0, 1, 2, 3, 4, 5, 6, 8, 9]])

gen_latents = torch.cat(samples)
# gen_latents = operator._unnorm(gen_latents)
# gen_latents = gen_latents[:,0]
print(f"Generated latents shape: {gen_latents.shape}")

Generated latents shape: torch.Size([10, 1, 128, 384])


In [8]:
# gen_latents = torch.load('Topography_unstructured_outputs/pts/generated_latents_sparsewell_and_co2seismic.pt')
nf_out_gene = []
for i in range(gen_latents.shape[0]):
    gen_latents_in = gen_latents[i].unsqueeze(0)
    nf_out_gene_i, _, _ = operator._cnf_forward(gen_latents_in)
    nf_out_gene.append(nf_out_gene_i.squeeze(0).detach().cpu())
nf_out_gene = torch.stack(nf_out_gene)
nf_out_gene = operator._unnorm_cnf(nf_out_gene)

# idxes = [0, 1, 2, 6, 8, 9]
# previous = torch.load('Topography_unstructured_outputs/pts/CNF_out_geoseismic.pt').cpu()[idxes, ...]
# nf_out_gene = torch.cat([previous, nf_out_gene.detach().cpu()[[0,1,3,4], ...]], dim=0)
# print(f"CNF output shape: {nf_out_gene.shape}")

torch.save(nf_out_gene, 'Topography_unstructured_outputs/pts/CNF_out_co2seismic_and_geoseismic_noise_0.05.pt')
np.save('Topography_unstructured_outputs/npys/CNF_out_co2seismic_and_geoseismic_noise_0.05.npy', nf_out_gene.detach().cpu().numpy())